In [1]:
import pandas as pd
from perplexity import Perplexity
from dotenv import load_dotenv
from typing import Optional, List

from pydantic import BaseModel
load_dotenv()

client = Perplexity()

In [5]:
dc_table = pd.read_csv('data_centers_table.csv')
dc_table_meta = pd.read_csv('data_centers_meta_data.csv')

In [ ]:
dc_table_meta.tail(1)

In [79]:
ids = list(dc_table_meta.dropna(subset=['overview'])['publicId'])

In [90]:


def create_query(id, dc_table_meta: pd.DataFrame = dc_table_meta, dc_table: pd.DataFrame =dc_table) -> str:
    row = dc_table_meta.set_index('publicId').loc[[id]]
    search_city = row['city'].values[0]
    search_country = row['country'].values[0]
    search_description = row['overview'].values[0]
    name = dc_table.set_index('publicId').loc[id].values[0]
    query = f"MW IT capacity, operation start date, Floor capacity, n buildings for: Data center in {search_city}, {search_country} - name: {name}. {search_description}. Company: GDS Holdings. Data Center ID is {id}."
    return query

In [86]:
class DataCenterInfo(BaseModel):
    publicId: str
    name: str
    company: str            
    location: str       
    country: Optional[str] = None 
    installed_power_capacity_mw: Optional[float] = None
    installed_power_capacity_mw_confidence: Optional[float] = None     
    operation_start_date: Optional[str] = None
    operation_start_date_confidence: Optional[float] = None
    installed_area_sq_ft: Optional[float] = None    
    installed_area_sq_ft_confidence: Optional[float] = None
    num_buildings: Optional[int] = None
    num_buildings_confidence: Optional[float] = None
    notes: Optional[str] = None
    sources: Optional[List[str]] = None

client = Perplexity()


In [88]:
target_id = ids[0]
dc_query = create_query(id=target_id)



In [111]:
concat_df= pd.DataFrame()


In [131]:


for id in ids[50:100]:
    print(f"Processing data center ID: {id}")
    dc_query = create_query(id=id)

    query = f"""
    {dc_query}\n
    From the web, extract those fields and fill the schema.
    If something is unknown, set it to null.
    """

    completion = client.chat.completions.create(
        model="sonar-pro",
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a research assistant that extracts structured data"
                    "about data centers from the web and also from the context given. "
                    "you're an expert in data centers."
                    "Return ONLY valid JSON that matches the schema — and include all URLs you used inside the `sources` array."
                    "All dates in YYYY-MM-DD format."
                ),
            },
            {"role": "user", "content": query},
        ],
        response_format={
            "type": "json_schema",
            "json_schema": {
                "schema": DataCenterInfo.model_json_schema()
            },
        },
    )

    data_center_model_data = DataCenterInfo.model_validate_json(completion.choices[0].message.content)
    dc_df = pd.DataFrame(data_center_model_data).set_index(0).T
    concat_df = pd.concat([concat_df, dc_df], ignore_index=True)

Processing data center ID: G32XlR
Processing data center ID: RN9PYB
Processing data center ID: Bb8YkG
Processing data center ID: Gamx6G
Processing data center ID: 9RYLdB
Processing data center ID: BgV7mw
Processing data center ID: woW8lw
Processing data center ID: RvXDaw
Processing data center ID: R8LXZR
Processing data center ID: B1pbVw
Processing data center ID: Rj09zw
Processing data center ID: Gl44gG
Processing data center ID: R8VxVw
Processing data center ID: GLjkAB
Processing data center ID: 8Rv0aw
Processing data center ID: Rj00Nw
Processing data center ID: w9q6pw
Processing data center ID: wVqKKG
Processing data center ID: Gy5pEG
Processing data center ID: GrljKB
Processing data center ID: GlP4zG
Processing data center ID: wVnLKw
Processing data center ID: B1Xm3w
Processing data center ID: B1jEQR
Processing data center ID: RNQXlB
Processing data center ID: RKv6eB
Processing data center ID: R4Dg8R
Processing data center ID: B1QvxB
Processing data center ID: 8Rv1AB
Processing dat

In [132]:
concat_df2 = concat_df.set_index('publicId')

In [133]:
meta_df = dc_table_meta.set_index('publicId')

In [134]:
meta_df['sources'] = None
meta_df['num_buildings'] = None

In [135]:
concat_df2[['installed_power_capacity_mw', 'operation_start_date', 'installed_area_sq_ft', 'num_buildings', 'sources']].tail()

,installed_power_capacity_mw,operation_start_date,installed_area_sq_ft,num_buildings,sources
publicId,,,,,
Gm4NJR,55.0,None,None,None,[https://w.media/doma-infrastructure-group-ann...
BPdAbB,128.0,None,None,None,[https://www.datacentermap.com/italy/milan/ntt...
RpZlqw,2.4,2023-01-01,26909.78,1,[https://www.datacentermap.com/germany/munich/...
kGakYB,6.0,2007-01-01,34983.0,1,[https://inflect.com/building/8435-north-stemm...
RpY5mG,None,None,None,5,"[https://www.gds-services.com/en/about_4.html,..."


In [140]:
meta_df.update(concat_df2[['installed_power_capacity_mw', 'operation_start_date', 'installed_area_sq_ft', 'num_buildings', 'sources']])

In [144]:
meta_df.to_csv('data_centers_metadata.csv')